# 04 — Inference: Pretrained vs Fine-Tuned YOLO26s

**Goal:** Compare COCO-pretrained YOLO26s with the BDD100K fine-tuned detector.

This notebook:
1. Loads both pretrained and fine-tuned YOLO26s weights
2. Runs inference on validation images with a fixed seed for reproducibility
3. Visualises and compares predictions side-by-side
4. Prints per-class detection counts and mean confidence statistics
5. Saves output images

## 1 — Setup

In [1]:
!pip install -q ultralytics>=8.4.0 opencv-python matplotlib

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import cv2
import random
import tarfile
import matplotlib.pyplot as plt
import numpy as np
import torch
from collections import defaultdict

# ── Reproducibility ────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Paths ──────────────────────────────────────────────────────────
ECOCAR_ROOT = "/content/drive/MyDrive/EcoCAR"
DATASET_DIR = "/content/bdd100k_yolo"

# Fine-tuned weights from notebook 03
WEIGHTS = os.path.join(ECOCAR_ROOT, "weights", "bdd100k_yolo26s_best.pt")

# Pretrained baseline — must be same architecture (yolo26s)
PRETRAINED_WEIGHTS = "yolo26s.pt"

# Confidence threshold for inference
CONF_THRES = 0.25

# Output directory
OUTPUT_DIR = os.path.join(ECOCAR_ROOT, "outputs", "04_inference")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Extract dataset if not already present ─────────────────────────
TAR_PATH = os.path.join(ECOCAR_ROOT, "datasets", "bdd100k_yolo_nb02.tar")
if not os.path.isdir(os.path.join(DATASET_DIR, "images", "val")):
    if os.path.isfile(TAR_PATH):
        os.makedirs(DATASET_DIR, exist_ok=True)
        print(f"Extracting {TAR_PATH} ...")
        with tarfile.open(TAR_PATH, "r") as tar:
            tar.extractall(DATASET_DIR)
        print("Extraction complete.")
    else:
        print(f"WARNING: Tar archive not found: {TAR_PATH}")
        print("Run Notebook 02 first. Falling back to sample images.")
else:
    print(f"Dataset already exists: {DATASET_DIR}")

print(f"Fine-tuned weights: {WEIGHTS}")
print(f"Pretrained weights: {PRETRAINED_WEIGHTS}")
print(f"Confidence thresh:  {CONF_THRES}")
print(f"Output dir:         {OUTPUT_DIR}")
print(f"Fine-tuned exists:  {os.path.isfile(WEIGHTS)}")

Extracting /content/drive/MyDrive/EcoCAR/datasets/bdd100k_yolo_nb02.tar ...


/tmp/ipykernel_7906/589226990.py:40: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(DATASET_DIR)


Extraction complete.
Fine-tuned weights: /content/drive/MyDrive/EcoCAR/weights/bdd100k_yolo26s_best.pt
Pretrained weights: yolo26s.pt
Confidence thresh:  0.25
Output dir:         /content/drive/MyDrive/EcoCAR/outputs/04_inference
Fine-tuned exists:  True


## 2 — Load Trained Model

In [4]:
from ultralytics import YOLO

# Load fine-tuned BDD100K model
ft_model = YOLO(WEIGHTS)

print(f"Fine-tuned model loaded: {WEIGHTS}")
print(f"  Task: {ft_model.task}")
print(f"  Classes ({len(ft_model.names)}):")
for idx, name in ft_model.names.items():
    print(f"    {idx}: {name}")

# Load pretrained COCO baseline (same architecture: yolo26s)
pre_model = YOLO(PRETRAINED_WEIGHTS)

print(f"\nPretrained model loaded: {PRETRAINED_WEIGHTS}")
print(f"  Task: {pre_model.task}")
print(f"  Classes: {len(pre_model.names)} (COCO)")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Fine-tuned model loaded: /content/drive/MyDrive/EcoCAR/weights/bdd100k_yolo26s_best.pt
  Task: detect
  Classes (10):
    0: person
    1: rider
    2: car
    3: truck
    4: bus
    5: train
    6: motorcycle
    7: bicycle
    8: traffic light
    9: traffic sign

Pretrained model loaded: yolo26s.pt
  Task: detect
  Classes: 80 (COCO)


## 3 — Select Inference Images

In [5]:
# Use validation images
val_images_dir = os.path.join(DATASET_DIR, "images", "val")

if os.path.isdir(val_images_dir):
    all_images = sorted([
        os.path.join(val_images_dir, f)
        for f in os.listdir(val_images_dir)
        if f.endswith(('.jpg', '.png', '.jpeg'))
    ])
    NUM_SAMPLES = 8
    sample_images = random.sample(all_images, min(NUM_SAMPLES, len(all_images)))
    print(f"Selected {len(sample_images)} images from {len(all_images)} val images (seed={SEED})")
else:
    print(f"Val images not found at {val_images_dir}")
    print("Falling back to sample images...")

    import urllib.request
    sample_dir = "sample_images"
    os.makedirs(sample_dir, exist_ok=True)
    urls = {
        "bus.jpg": "https://ultralytics.com/images/bus.jpg",
        "zidane.jpg": "https://ultralytics.com/images/zidane.jpg",
    }
    for name, url in urls.items():
        path = os.path.join(sample_dir, name)
        if not os.path.exists(path):
            urllib.request.urlretrieve(url, path)
    sample_images = sorted([os.path.join(sample_dir, f) for f in os.listdir(sample_dir)])

for p in sample_images:
    print(f"  {os.path.basename(p)}")

Selected 8 images from 10000 val images (seed=42)
  b63b1c14-f01a2f6b.jpg
  b2b70230-bad4ff6e.jpg
  bd50478a-4eb79ba3.jpg
  bc07d865-b2971089.jpg
  bb2f678a-af43ed71.jpg
  b764f1b2-7ce1c79f.jpg
  b5dd20c5-ed2da652.jpg
  c7dab685-05c2f1df.jpg


## 4 — Run Inference

In [6]:
# Run inference on all sample images with confidence threshold
ft_results = ft_model(sample_images, conf=CONF_THRES, verbose=False)

print(f"Inference complete on {len(ft_results)} images (conf>={CONF_THRES})")
total_dets = sum(len(r.boxes) for r in ft_results)
print(f"Total detections: {total_dets}")

Inference complete on 8 images (conf>=0.25)
Total detections: 116


## 5 — Visualise Predictions

In [7]:
# Display fine-tuned predictions in a grid
n_cols = 2
n_rows = (len(ft_results) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 6 * n_rows))
if n_rows == 1:
    axes = axes[np.newaxis, :] if n_cols > 1 else np.array([[axes]])
if n_cols == 1:
    axes = axes[:, np.newaxis]

for i, result in enumerate(ft_results):
    row, col = i // n_cols, i % n_cols
    annotated = result.plot()
    axes[row, col].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    axes[row, col].set_title(
        f"{os.path.basename(sample_images[i])} -- {len(result.boxes)} detections",
        fontsize=9,
    )
    axes[row, col].axis('off')

for i in range(len(ft_results), n_rows * n_cols):
    row, col = i // n_cols, i % n_cols
    axes[row, col].axis('off')

plt.suptitle('BDD100K Fine-Tuned YOLO26s Predictions', fontsize=14)
plt.tight_layout()
plt.show()

Output hidden; open in https://colab.research.google.com to view.

## 6 — Save Output Images

In [8]:
for i, result in enumerate(ft_results):
    annotated = result.plot()
    save_path = os.path.join(OUTPUT_DIR, f"pred_{os.path.basename(sample_images[i])}")
    cv2.imwrite(save_path, annotated)
    print(f"  Saved: {save_path}")

print(f"\nAll predictions saved to: {OUTPUT_DIR}")

  Saved: /content/drive/MyDrive/EcoCAR/outputs/04_inference/pred_b63b1c14-f01a2f6b.jpg
  Saved: /content/drive/MyDrive/EcoCAR/outputs/04_inference/pred_b2b70230-bad4ff6e.jpg
  Saved: /content/drive/MyDrive/EcoCAR/outputs/04_inference/pred_bd50478a-4eb79ba3.jpg
  Saved: /content/drive/MyDrive/EcoCAR/outputs/04_inference/pred_bc07d865-b2971089.jpg
  Saved: /content/drive/MyDrive/EcoCAR/outputs/04_inference/pred_bb2f678a-af43ed71.jpg
  Saved: /content/drive/MyDrive/EcoCAR/outputs/04_inference/pred_b764f1b2-7ce1c79f.jpg
  Saved: /content/drive/MyDrive/EcoCAR/outputs/04_inference/pred_b5dd20c5-ed2da652.jpg
  Saved: /content/drive/MyDrive/EcoCAR/outputs/04_inference/pred_c7dab685-05c2f1df.jpg

All predictions saved to: /content/drive/MyDrive/EcoCAR/outputs/04_inference


## 7 — Summary Statistics

In [9]:
# ── Per-class detection counts and mean confidence ─────────────────
class_counts = defaultdict(int)
class_confs  = defaultdict(list)

for result in ft_results:
    for box in result.boxes:
        cls_id = int(box.cls.item())
        cls_name = ft_model.names[cls_id]
        class_counts[cls_name] += 1
        class_confs[cls_name].append(box.conf.item())

print(f"{'Class':<20} {'Count':>6} {'Mean Conf':>10} {'Min':>7} {'Max':>7}")
print("-" * 52)
for cls_name in sorted(class_counts, key=lambda c: -class_counts[c]):
    confs = class_confs[cls_name]
    print(
        f"{cls_name:<20} {class_counts[cls_name]:>6} "
        f"{np.mean(confs):>10.3f} {min(confs):>7.3f} {max(confs):>7.3f}"
    )

total = sum(class_counts.values())
all_confs = [c for cs in class_confs.values() for c in cs]
print("-" * 52)
print(f"{'TOTAL':<20} {total:>6} {np.mean(all_confs):>10.3f} {min(all_confs):>7.3f} {max(all_confs):>7.3f}")

Class                 Count  Mean Conf     Min     Max
----------------------------------------------------
car                      54      0.630   0.253   0.949
traffic sign             31      0.582   0.257   0.849
traffic light            25      0.520   0.270   0.838
truck                     5      0.551   0.257   0.821
person                    1      0.286   0.286   0.286
----------------------------------------------------
TOTAL                   116      0.587   0.253   0.949


## 8 — Side-by-Side: Pretrained YOLO26s vs Fine-Tuned

Compare the COCO-pretrained YOLO26s (same architecture) with the BDD100K fine-tuned model.

In [10]:
# Pick first 4 images for comparison
compare_images = sample_images[:4]

for img_path in compare_images:
    # Pretrained results (same arch: yolo26s)
    pre_result = pre_model(img_path, conf=CONF_THRES, verbose=False)[0]
    pre_annotated = pre_result.plot()

    # Fine-tuned results
    ft_result = ft_model(img_path, conf=CONF_THRES, verbose=False)[0]
    ft_annotated = ft_result.plot()

    # Side by side
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))
    ax1.imshow(cv2.cvtColor(pre_annotated, cv2.COLOR_BGR2RGB))
    ax1.set_title(f"COCO Pretrained yolo26s ({len(pre_result.boxes)} dets)", fontsize=12)
    ax1.axis('off')
    ax2.imshow(cv2.cvtColor(ft_annotated, cv2.COLOR_BGR2RGB))
    ax2.set_title(f"BDD100K Fine-Tuned yolo26s ({len(ft_result.boxes)} dets)", fontsize=12)
    ax2.axis('off')
    plt.suptitle(os.path.basename(img_path), fontsize=13)
    plt.tight_layout()

    save_path = os.path.join(OUTPUT_DIR, f"compare_{os.path.basename(img_path)}")
    plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f"  Saved: {save_path}")

Output hidden; open in https://colab.research.google.com to view.

## 9 — How to Reuse Trained Weights

```python
from ultralytics import YOLO

# Load your fine-tuned model
model = YOLO("/content/drive/MyDrive/EcoCAR/weights/bdd100k_yolo26s_best.pt")

# Run inference with confidence threshold
results = model("path/to/image.jpg", conf=0.25)

# Access predictions
for result in results:
    for box in result.boxes:
        class_name = model.names[int(box.cls)]
        confidence = box.conf.item()
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
```

In [11]:
print("\n" + "=" * 60)
print(" INFERENCE SUMMARY")
print("=" * 60)
print(f" Fine-tuned model: {WEIGHTS}")
print(f" Pretrained model: {PRETRAINED_WEIGHTS}")
print(f" Conf threshold:   {CONF_THRES}")
print(f" Images:           {len(ft_results)}")
print(f" Total detections: {sum(len(r.boxes) for r in ft_results)}")
print(f" Unique classes:   {len(class_counts)}")
print(f" Saved to:         {OUTPUT_DIR}")
print("=" * 60)
print("\nNext: notebook 05 (feature extraction) or 06 (model inspection)")


 INFERENCE SUMMARY
 Fine-tuned model: /content/drive/MyDrive/EcoCAR/weights/bdd100k_yolo26s_best.pt
 Pretrained model: yolo26s.pt
 Conf threshold:   0.25
 Images:           8
 Total detections: 116
 Unique classes:   5
 Saved to:         /content/drive/MyDrive/EcoCAR/outputs/04_inference

Next: notebook 05 (feature extraction) or 06 (model inspection)
